In [1]:
"""
data/dataset.py
===============
KiTS21 dataset.  Returns:
  - train mode : 16-slice tumour-centred crop  (3, D=16, H, W)
  - val   mode : full volume                   (3, D,    H, W)

Each sample always includes event flag and survival time for downstream
survival training.
"""

import json
import os
import random

import numpy as np
import SimpleITK as sitk
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset
from torchvision.transforms import InterpolationMode


class KitsDataset(Dataset):
    def __init__(
        self,
        rootdir:        str,
        target_spacing: tuple,          # (X, Y, Z) SimpleITK convention
        target_shape:   tuple,          # (H, W) after spatial resize
        split_file:     str,
        metadata_path:  str,
        p:              float = 0.8,    # prob of tumour-centred depth crop
        mode:           str   = "train",
        crop_depth:     int   = 16,
    ):
        self.rootdir        = rootdir
        self.target_spacing = target_spacing
        self.target_shape   = target_shape
        self.split_file     = split_file
        self.metadata_path  = metadata_path
        self.p              = p
        self.mode           = mode
        self.crop_depth     = crop_depth

        self.metadata: dict = {}
        self._load_cases()

    # ── Initialisation ────────────────────────────────────────────────────

    def _load_cases(self) -> None:
        with open(self.split_file, "r") as f:
            splits = json.load(f)
        self.cases = splits.get(self.mode, splits.get("train"))

        with open(self.metadata_path, "r") as f:
            for entry in json.load(f):
                self.metadata[entry["case_id"]] = entry

    # ── Dataset protocol ──────────────────────────────────────────────────

    def __len__(self) -> int:
        return len(self.cases)

    def __getitem__(self, index: int) -> dict:
        caseid = self.cases[index]

        image = sitk.ReadImage(os.path.join(self.rootdir, caseid, "imaging.nii.gz"))
        mask  = sitk.ReadImage(os.path.join(self.rootdir, caseid, "aggregated_MAJ_seg.nii.gz"))

        image = sitk.DICOMOrient(image, "RAS")
        mask  = sitk.DICOMOrient(mask,  "RAS")

        image, mask = self._resample(image, mask)
        image, mask = self._to_tensors(image, mask)
        image, mask = self._spatial_resize(image, mask)  # H, W → target_shape

        if self.mode == "train":
            image, mask = self._train_crop(image, mask)  # (3, crop_depth, H, W)
        # val: full volume (3, D, H, W) — sliding window handled in training loop

        event, survival_time = self._get_survival(caseid)

        return {
            "ct":            image,
            "mask":          mask,
            "caseid":        caseid,
            "event":         torch.tensor(event,         dtype=torch.bool),
            "survival_time": torch.tensor(survival_time, dtype=torch.float32),
        }

    # ── Private helpers ───────────────────────────────────────────────────

    def _resample(
        self,
        image: sitk.Image,
        mask:  sitk.Image,
    ) -> tuple[sitk.Image, sitk.Image]:
        original_size    = image.GetSize()
        original_spacing = image.GetSpacing()

        new_size = [
            int(round(osz * osp / tsp))
            for osz, osp, tsp in zip(original_size, original_spacing, self.target_spacing)
        ]

        def _do_resample(itk_img: sitk.Image, is_mask: bool) -> sitk.Image:
            r = sitk.ResampleImageFilter()
            r.SetSize(new_size)
            r.SetOutputSpacing(self.target_spacing)
            r.SetOutputOrigin(itk_img.GetOrigin())
            r.SetOutputDirection(itk_img.GetDirection())
            if is_mask:
                r.SetInterpolator(sitk.sitkNearestNeighbor)
                r.SetDefaultPixelValue(0)
            else:
                r.SetInterpolator(sitk.sitkLinear)
                r.SetDefaultPixelValue(-1000)
            return r.Execute(itk_img)

        return _do_resample(image, False), _do_resample(mask, True)

    def _to_tensors(
        self,
        image: sitk.Image,
        mask:  sitk.Image,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        img_arr  = sitk.GetArrayFromImage(image).astype(np.float32)  # (D, H, W)
        mask_arr = sitk.GetArrayFromImage(mask).astype(np.int64)     # (D, H, W)

        def apply_window(arr: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
            """Clips HU values to a specific window and normalizes to [0, 1]."""
            clipped = np.clip(arr, vmin, vmax)
            return (clipped - vmin) / (vmax - vmin)

        # Channel 0 (Red): Soft Tissue & Tumor (-100 to 250 HU)
        ch_red = apply_window(img_arr, -100.0, 250.0)
        
        # Channel 1 (Green): Fat & Boundary (-150 to -30 HU)
        ch_green = apply_window(img_arr, -150.0, -30.0)
        
        # Channel 2 (Blue): Vascular & Enhancement (100 to 400 HU)
        ch_blue = apply_window(img_arr, 100.0, 400.0)

        # Stack into a 3-channel RGB array: shape becomes (3, D, H, W)
        rgb_arr = np.stack([ch_red, ch_green, ch_blue], axis=0)

        return torch.from_numpy(rgb_arr), torch.from_numpy(mask_arr)

    def _spatial_resize(
        self,
        image: torch.Tensor,   # (3, D, H, W)
        mask:  torch.Tensor,   # (D, H, W) int64
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # Image is 4D, TF.resize handles the last two dimensions (H, W) automatically
        image = TF.resize(
            image,                     
            list(self.target_shape),
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        )

        mask = TF.resize(
            mask.float().unsqueeze(0),
            list(self.target_shape),
            interpolation=InterpolationMode.NEAREST,
        ).squeeze(0).long()

        return image, mask

    def _train_crop(
        self,
        image: torch.Tensor,   # (3, D, H, W)
        mask:  torch.Tensor,   # (D, H, W)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        depth = image.shape[1]
        crop  = self.crop_depth

        nonzero_slices = (mask != 0).any(dim=(1, 2))
        indices        = torch.where(nonzero_slices)[0]

        if len(indices) > 0 and torch.rand(()) < self.p:
            center = indices[random.randint(0, len(indices) - 1)].item()
            z_min  = max(0, center - crop + 1)
            z_max  = min(center, depth - crop)
            z = (
                max(0, min(center, depth - crop))
                if z_max < z_min
                else random.randint(z_min, z_max)
            )
        else:
            z = random.randint(0, max(0, depth - crop))

        return image[:, z:z + crop], mask[z:z + crop]

    def _get_survival(self, caseid: str) -> tuple[bool, float]:
        """
        Returns
        -------
        event         : True if patient died (vital_status == 'dead')
        survival_time : days after surgery (vital_days_after_surgery)
        """
        meta          = self.metadata[caseid]
        event         = meta["vital_status"] == "dead"
        survival_time = meta.get("vital_days_after_surgery") or 0.0
        return bool(event), float(survival_time)

In [2]:
from configs.survival_config import SurvivalConfig
import os
cfg    = SurvivalConfig()
_ds_kwargs = dict(
        rootdir        = cfg.root_dir,
        target_spacing = cfg.target_spacing,
        target_shape   = cfg.target_shape,
        split_file     = cfg.json_path,
        metadata_path  = os.path.join(cfg.root_dir, "kits.json"),
    )
train_ds = KitsDataset(**_ds_kwargs, mode="train")

In [13]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
from torch.utils.data import DataLoader
from IPython.display import display

# Assuming train_ds is already initialized
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)

# Fetch a single random batch
batch = next(iter(train_loader))

# Note: The shape is now (3, 16, 256, 256) due to the 3 RGB channels
ct_tensor = batch["ct"][0].numpy()       
mask_tensor = batch["mask"][0].numpy()   # Shape: (16, 256, 256)
case_id = batch["caseid"][0]

print(f"Loaded Case: {case_id}")
print(f"CT Tensor Shape: {ct_tensor.shape}")
print(f"Mask Tensor Shape: {mask_tensor.shape}")


Loaded Case: case_00292
CT Tensor Shape: (3, 16, 256, 256)
Mask Tensor Shape: (16, 256, 256)


In [14]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
from torch.utils.data import DataLoader
from IPython.display import display

# Assuming train_ds and dataloader are already initialized
# train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
# batch = next(iter(train_loader))
# ct_tensor = batch["ct"][0].numpy()       # Shape: (3, 16, 256, 256)
# mask_tensor = batch["mask"][0].numpy()   # Shape: (16, 256, 256)

def visualize_single_channel(z, channel):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Map the dropdown selection to the correct channel index
    if channel == 'Red (Soft Tissue: -100 to 250 HU)':
        c_idx = 0
    elif channel == 'Green (Fat & Boundary: -150 to -30 HU)':
        c_idx = 1
    elif channel == 'Blue (Vascular: 100 to 400 HU)':
        c_idx = 2
        
    # Extract the 2D slice for the chosen channel and Z-depth
    # ct_tensor shape: (3, D, H, W) -> slice shape becomes (H, W)
    channel_slice = ct_tensor[c_idx, z, :, :]
    
    # Plot the single channel window (using grayscale for clear clinical inspection)
    axes[0].imshow(channel_slice, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f"{channel}\nZ-Slice: {z}")
    axes[0].axis("off")
    
    # Plot Segmentation Mask
    slice_mask = mask_tensor[z]
    axes[1].imshow(slice_mask, cmap='nipy_spectral', interpolation='nearest')
    axes[1].set_title(f"Segmentation Mask (Z={z})")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

# Create the interactive widgets
depth = ct_tensor.shape[1]
z_slider = widgets.IntSlider(
    min=0, max=depth - 1, value=depth // 2, description='Z-Slice'
)

channel_dropdown = widgets.Dropdown(
    options=[
        'Red (Soft Tissue: -100 to 250 HU)', 
        'Green (Fat & Boundary: -150 to -30 HU)', 
        'Blue (Vascular: 100 to 400 HU)'
    ],
    value='Red (Soft Tissue: -100 to 250 HU)',
    description='Channel:'
)

# Bind the widgets to the visualization function
widgets.interact(visualize_single_channel, z=z_slider, channel=channel_dropdown)

interactive(children=(IntSlider(value=8, description='Z-Slice', max=15), Dropdown(description='Channel:', opti…

<function __main__.visualize_single_channel(z, channel)>